# Differential Expression Analysis

HepG2 세포에서 세 조건(control / SIRT6 WT / SIRT6 variant N308K/A313S)을 비교합니다.

| 비교 | 질문 |
|---|---|
| **wt vs control** | SIRT6 과발현 자체의 효과 |
| **variant vs control** | 장수 변이 SIRT6의 효과 |
| **variant vs wt** | 변이가 WT와 *다르게* 하는 것 (핵심 질문) |

log2FC > 0 → 앞쪽 조건(numerator)에서 발현이 높음.

### 방법
- **메인: DESeq2 (`pydeseq2`)** — raw count에 negative binomial 모델, median-of-ratios 정규화, dispersion shrinkage. replicate가 3개뿐일 때 t-test보다 훨씬 안정적입니다.
- **비교용: Welch t-test on log2 CPM + BH FDR** — README에 적어둔 원래 계획. 두 방법의 결과가 얼마나 겹치는지 확인합니다.

입력: `02_normalization_qc.ipynb`에서 저장한 `data/processed/counts_filtered.csv` (저발현 유전자 제거된 **raw count**)

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.normalization import load_metadata, log2_cpm, ensure_dir
from src.stats import CONTRASTS, run_deseq2, deseq2_contrast, ttest_de, call_degs, deg_summary
from src.plotting import volcano_plot, ma_plot

PADJ, LFC = 0.05, 1.0   # DEG 기준: padj < 0.05 & |log2FC| > 1
TABLES = ensure_dir("../results/tables")
FIGS = ensure_dir("../results/figures")

## 1. 데이터 불러오기

In [ ]:
counts = pd.read_csv("../data/processed/counts_filtered.csv", index_col=0)
meta = load_metadata("../data/raw/metadata.csv", counts=counts)

# control을 reference level로
meta["condition"] = pd.Categorical(meta["condition"], categories=["control", "wt", "variant"])

print(counts.shape)
meta

## 2. DESeq2 (pydeseq2)

9개 샘플 전체로 모델을 한 번 fitting 하고, 세 contrast에 대해 Wald test를 합니다.
(몇 분 걸릴 수 있습니다.)

In [ ]:
dds = run_deseq2(counts, meta, n_cpus=4)

# DESeq2 size factor (median of ratios)
dds.obs["size_factors"].round(3)

In [ ]:
deseq_res = {}
for num, den in CONTRASTS:
    name = f"{num}_vs_{den}"
    res = deseq2_contrast(dds, num, den, n_cpus=4)
    res["regulation"] = call_degs(res, PADJ, LFC)
    res.to_csv(TABLES / f"deseq2_{name}.csv")
    deseq_res[name] = res

deseq_summary = deg_summary(deseq_res, PADJ, LFC)
deseq_summary.to_csv(TABLES / "deseq2_deg_summary.csv")
deseq_summary

### 비교별 상위 DEG

In [ ]:
for name, res in deseq_res.items():
    print(f"===== {name} =====")
    display(res[res["regulation"] != "ns"].head(10)[["baseMean", "log2FoldChange", "padj", "regulation"]])

## 3. Volcano / MA plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, res) in zip(axes, deseq_res.items()):
    volcano_plot(res, title=name, padj=PADJ, lfc=LFC, ax=ax, label_top=0)
plt.tight_layout()
fig.savefig(FIGS / "volcano_deseq2.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (name, res) in zip(axes, deseq_res.items()):
    ma_plot(res, title=name, padj=PADJ, ax=ax)
plt.tight_layout()
fig.savefig(FIGS / "ma_deseq2.png", dpi=150)
plt.show()

## 4. 비교용: Welch t-test (log2 CPM) + BH FDR

In [ ]:
log_cpm = log2_cpm(counts)

ttest_res = {}
for num, den in CONTRASTS:
    name = f"{num}_vs_{den}"
    res = ttest_de(log_cpm, meta, num, den)
    res["regulation"] = call_degs(res, PADJ, LFC)
    res.to_csv(TABLES / f"ttest_{name}.csv")
    ttest_res[name] = res

deg_summary(ttest_res, PADJ, LFC)

In [ ]:
# 두 방법의 DEG 겹침
rows = {}
for name in deseq_res:
    d = set(deseq_res[name].index[deseq_res[name]["regulation"] != "ns"])
    t = set(ttest_res[name].index[ttest_res[name]["regulation"] != "ns"])
    rows[name] = {"DESeq2": len(d), "t-test": len(t), "both": len(d & t)}
pd.DataFrame(rows).T

> 그룹당 n=3이라 t-test는 검정력이 낮고 분산 추정이 불안정합니다. 결과 해석은 DESeq2를 기준으로 합니다.

## 5. 메모 / 다음 단계

- `04_visualization.ipynb`: DEG heatmap, 세 비교의 DEG 겹침, Ensembl ID → gene symbol 변환
- variant_vs_wt DEG 중 침윤/전이(EMT, migration) 관련 유전자 확인
- GO / pathway enrichment (예: `gseapy`)
- 결과 요약 → `reports/analysis_summary.md`